# Feature pruning: does the model need all 66 features?

The feature store has 66 features that need to be tracked by monitoring every month. This notebook tests whether the models lose anything without the features neither candidate uses, under a rule written before any comparison is run.

The feature store itself does not change. The pruned list only changes which columns `model_train.py` reads.

## 1. Decision rule (written before the comparison)

Which features to drop is decided on the training split only. At each of three train dates (2024-01-01, 2024-03-01 and 2024-05-01), using that date's training split:
1. Tune each candidate (logistic regression and XGBoost) on all 66 features, with the pipeline's search.
2. For each of the 3 cross-validation folds, refit the tuned candidate on the other two folds. In the held-out fold, shuffle one feature at a time (10 shuffles each) and measure the AUC lost.
3. A categorical's one-hot columns are shuffled together, as one feature, so a category is kept or dropped whole: Occupation, Credit_Mix, Payment_of_Min_Amount and Payment_Behaviour. Every other column is shuffled alone, including each clickstream feature. That makes 43 features.
4. A feature is dropped if its mean AUC loss is 0 or less for both candidates: neither model does worse without it.

This gives one list per train date, used by both candidates. The pipeline would use the 2024-03-01 list.

Keep the smaller list only if the selected model loses no more than 0.005 OOT AUC at each of the three train dates.
- The selected model is whichever candidate has the higher OOT AUC, compared with the 66-feature model selected the same way.
- 0.005 is the bar notebook 03 set for adding features, so adding and removing face the same test.
- It is half the ±0.01 that one OOT window's AUC moves by from sampling alone.

OOT is used once, for this check. Bootstrap intervals are reported to show how much of each change is noise, but they are not part of the rule. On about 950 applications, a paired interval is roughly ±0.007 wide, so it cannot show that two models are within 0.005 of each other.

In [1]:
import json
import textwrap

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta

from sklearn.base import clone
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import xgboost as xgb

pd.set_option("display.width", 200)

In [2]:
# gold stores built by the DAG
features = pd.read_parquet("../datamart/gold/feature_store/")
labels = pd.read_parquet("../datamart/gold/label_store/")
for df in (features, labels):
    df["snapshot_date"] = pd.to_datetime(df["snapshot_date"])
labels["loan_start_date"] = pd.to_datetime(labels["loan_start_date"])
print("features", features.shape, "labels", labels.shape)

features (12500, 68) labels (12500, 6)


## 2. Same training as the pipeline

As in `model_train.py` and notebook 03:
- windows on the application month: 6 months train/test and 2 months OOT, with every label dated before the train date;
- rows sorted, then a stratified 80/20 split;
- median imputation and scaling fitted on the training split;
- a random search with 3-fold cross-validation on AUC.

The first check: at 2024-03-01, all 66 features must give the pipeline's numbers in `model_bank/`.

In [3]:
# join features to labels on customer + application month, as in notebook 02
data = features.merge(
    labels[["Customer_ID", "loan_start_date", "label", "snapshot_date"]].rename(columns={"loan_start_date": "app_month", "snapshot_date": "label_date"}),
    left_on=["Customer_ID", "snapshot_date"], right_on=["Customer_ID", "app_month"], how="inner")
data = data.sort_values(["snapshot_date", "Customer_ID"]).reset_index(drop=True)
ALL_FEATURES = [c for c in features.columns if c not in ("Customer_ID", "snapshot_date")]
CHECK_DATES = ["2024-01-01", "2024-03-01", "2024-05-01"]
MODELS = ["logreg", "xgb"]

PARAM_DIST = {
    "logreg": {"C": np.logspace(-3, 2, 20)},
    "xgb": {"n_estimators": [25, 50, 100], "max_depth": [2, 3, 4], "learning_rate": [0.01, 0.1],
            "subsample": [0.6, 0.8], "colsample_bytree": [0.6, 0.8], "min_child_weight": [1, 3, 5]},
}

def train(cols, train_date, model_type):
    T = pd.Timestamp(train_date)
    oot_end = T - relativedelta(months=7)  # label known 6 months after application, and dated before T
    oot_start = oot_end - relativedelta(months=1)
    tt_start = oot_start - relativedelta(months=6)
    rows = data[(data["label_date"] < T) & (data["snapshot_date"] >= tt_start) & (data["snapshot_date"] <= oot_end)]
    oot = rows[rows["snapshot_date"] >= oot_start]
    train_test = rows[rows["snapshot_date"] < oot_start]
    X_train, X_test, y_train, y_test = train_test_split(train_test[cols], train_test["label"], test_size=0.2, random_state=88, shuffle=True, stratify=train_test["label"])
    imputer = SimpleImputer(strategy="median").fit(X_train)
    scaler = StandardScaler().fit(imputer.transform(X_train))
    prep = lambda X: scaler.transform(imputer.transform(X))
    estimator = LogisticRegression(max_iter=1000, random_state=88) if model_type == "logreg" else xgb.XGBClassifier(eval_metric="logloss", random_state=88)
    search = RandomizedSearchCV(estimator, PARAM_DIST[model_type], scoring="roc_auc", n_iter=20, cv=3, random_state=42, n_jobs=-1)
    model = search.fit(prep(X_train), y_train).best_estimator_
    oot_scores = model.predict_proba(prep(oot[cols]))[:, 1]
    return {"model": model, "X_train": X_train, "y_train": y_train, "y_oot": oot["label"].values, "oot_scores": oot_scores,
            "cv": search.best_score_, "oot": roc_auc_score(oot["label"], oot_scores)}

full = {(d, m): train(ALL_FEATURES, d, m) for d in CHECK_DATES for m in MODELS}
record = json.load(open("../model_bank/model_selection_2024_03_01.json"))
pd.DataFrame([{"model": m, "notebook OOT AUC": full[("2024-03-01", m)]["oot"],
               "model_bank OOT AUC": record["candidates"][f"credit_model_{m}_2024_03_01"]} for m in MODELS]
             ).assign(equal=lambda df: df["notebook OOT AUC"] == df["model_bank OOT AUC"])

,model,notebook OOT AUC,model_bank OOT AUC,equal
0,logreg,0.840153,0.840153,True
1,xgb,0.856338,0.856338,True


## 3. Which features does neither model use? (training split only)

In [4]:
CATEGORICALS = ["Occupation", "Credit_Mix", "Payment_of_Min_Amount", "Payment_Behaviour"]

def feature_groups(cols):
    """a categorical's one-hot columns form one feature; every other column is its own"""
    groups = {}
    for c in cols:
        groups.setdefault(next((cat for cat in CATEGORICALS if c.startswith(cat + "_")), c), []).append(c)
    return groups

def permutation_loss(run, groups, n_repeats=10, seed=0):
    """mean AUC lost on the held-out fold when a feature's columns are shuffled, over the 3 folds of the training split"""
    X, y = run["X_train"], run["y_train"]
    rng = np.random.default_rng(seed)
    losses = {g: [] for g in groups}
    for fit_rows, held_rows in StratifiedKFold(3).split(X, y):
        model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), clone(run["model"])).fit(X.iloc[fit_rows], y.iloc[fit_rows])
        X_held, y_held = X.iloc[held_rows], y.iloc[held_rows]
        base = roc_auc_score(y_held, model.predict_proba(X_held)[:, 1])
        for g, cols in groups.items():
            for _ in range(n_repeats):
                X_shuffled = X_held.copy()
                X_shuffled[cols] = X_held[cols].values[rng.permutation(len(X_held))]
                losses[g].append(base - roc_auc_score(y_held, model.predict_proba(X_shuffled)[:, 1]))
    return pd.Series({g: np.mean(v) for g, v in losses.items()})

groups = feature_groups(ALL_FEATURES)
loss = {(d, m): permutation_loss(full[(d, m)], groups) for d in CHECK_DATES for m in MODELS}
dropped = {d: [g for g in groups if loss[(d, "logreg")][g] <= 0 and loss[(d, "xgb")][g] <= 0] for d in CHECK_DATES}
kept_cols = {d: [c for c in ALL_FEATURES if not any(c in groups[g] for g in dropped[d])] for d in CHECK_DATES}
pd.DataFrame([{"train date": d, "features": len(groups), "columns": len(ALL_FEATURES), "features dropped": len(dropped[d]),
               "columns kept": len(kept_cols[d])} for d in CHECK_DATES])

,train date,features,columns,features dropped,columns kept
0,2024-01-01,43,66,9,55
1,2024-03-01,43,66,7,59
2,2024-05-01,43,66,7,45


In [5]:
# which features are dropped at which date
ever_dropped = [g for g in groups if any(g in dropped[d] for d in CHECK_DATES)]
pd.DataFrame({d: ["dropped" if g in dropped[d] else "" for g in ever_dropped] for d in CHECK_DATES}, index=ever_dropped)

,2024-01-01,2024-03-01,2024-05-01
Age,,dropped,
Annual_Income,dropped,,
Monthly_Inhand_Salary,dropped,,dropped
Num_Bank_Accounts,dropped,dropped,
Credit_Utilization_Ratio,,,dropped
Monthly_Balance,,dropped,
Occupation,,,dropped
Credit_Mix,dropped,,
fe_1_avg_3m,,,dropped
fe_6_avg_3m,,dropped,dropped


In [6]:
# 2024-03-01, the date the pipeline trains at: AUC lost when each feature is shuffled, smallest first
table = pd.DataFrame({m: loss[("2024-03-01", m)] for m in MODELS})
table["dropped"] = table.index.isin(dropped["2024-03-01"])
table.sort_values("xgb").round(4).head(16)

,logreg,xgb,dropped
fe_16_avg_3m,-0.0018,-0.0013,True
fe_6_avg_3m,-0.0002,-0.0008,True
fe_20_avg_3m,-0.0002,-0.0007,True
Monthly_Inhand_Salary,0.0004,-0.0004,False
Monthly_Balance,-0.0000,-0.0003,True
Credit_History_Months,0.0007,-0.0002,False
fe_13_avg_3m,0.0001,-0.0002,False
Num_of_Loan,0.0032,-0.0001,False
fe_17_avg_3m,-0.0000,-0.0000,True
Age,-0.0000,-0.0000,True


**Findings:**
- At 2024-03-01, shuffling 7 of the 43 features costs neither model any AUC: Age, Num_Bank_Accounts, Monthly_Balance and four clickstream features (fe_6, fe_16, fe_17 and fe_20). Dropping them leaves 59 of the 66 columns, and all four categoricals are kept.
- The other dates drop 9 and 7 features. At 2024-05-01 that includes Occupation, which is 15 columns, so 45 are left.
- No feature is dropped at all three dates. These features' losses are within about ±0.002 AUC of zero, so which side of zero each falls on changes from one training window to the next. The list is a finding at one date, not a fixed property of the features.

## 4. OOT check (run once)

In [7]:
pruned = {(d, m): train(kept_cols[d], d, m) for d in CHECK_DATES for m in MODELS}
oot = pd.DataFrame([{"train date": d, "model": m, "columns kept": len(kept_cols[d]),
                     "CV AUC (66)": full[(d, m)]["cv"], "CV AUC (kept)": pruned[(d, m)]["cv"],
                     "OOT AUC (66)": full[(d, m)]["oot"], "OOT AUC (kept)": pruned[(d, m)]["oot"]}
                    for d in CHECK_DATES for m in MODELS])
oot["OOT change"] = oot["OOT AUC (kept)"] - oot["OOT AUC (66)"]
oot.round(4)

,train date,model,columns kept,CV AUC (66),CV AUC (kept),OOT AUC (66),OOT AUC (kept),OOT change
0,2024-01-01,logreg,55,0.7888,0.7966,0.8308,0.8392,0.0084
1,2024-01-01,xgb,55,0.7964,0.8066,0.8461,0.8478,0.0017
2,2024-03-01,logreg,59,0.8122,0.8164,0.8402,0.8410,0.0008
3,2024-03-01,xgb,59,0.8279,0.8316,0.8563,0.8558,-0.0005
4,2024-05-01,logreg,45,0.8330,0.8388,0.8456,0.8517,0.0061
5,2024-05-01,xgb,45,0.8380,0.8430,0.8470,0.8418,-0.0053


In [8]:
# the model we would deploy: the higher OOT AUC, with all 66 features and with the kept list
def selected(runs, d):
    m = max(MODELS, key=lambda m: runs[(d, m)]["oot"])
    return m, runs[(d, m)]["oot"]

change = pd.DataFrame([{"train date": d, "selected (66)": selected(full, d)[0], "OOT AUC (66)": selected(full, d)[1],
                        "selected (kept)": selected(pruned, d)[0], "OOT AUC (kept)": selected(pruned, d)[1]} for d in CHECK_DATES])
change["change"] = change["OOT AUC (kept)"] - change["OOT AUC (66)"]
change.round(4)

,train date,selected (66),OOT AUC (66),selected (kept),OOT AUC (kept),change
0,2024-01-01,xgb,0.8461,xgb,0.8478,0.0017
1,2024-03-01,xgb,0.8563,xgb,0.8558,-0.0005
2,2024-05-01,xgb,0.8470,logreg,0.8517,0.0047


**Findings:**
- The selected model's OOT AUC changes by +0.0017, −0.0005 and +0.0047. That is within the 0.005 tolerance at all three dates.
- Logistic regression gains at every date (+0.0084, +0.0008, +0.0061). XGBoost changes by +0.0017, −0.0005 and −0.0053.
- At 2024-05-01, the pruned XGBoost loses slightly more than the tolerance. But logistic regression with the kept list becomes the selected model there (0.8517), so the model we would deploy still gains.
- Cross-validated AUC on the training split is higher with the kept list for both candidates at every date. For example, XGBoost at 2024-03-01 goes from 0.8279 to 0.8316.

## 5. How much of each change is noise?

The same bootstrap as notebook 03: draw the OOT applications with replacement 1,000 times, compute both models' AUC on each draw, and take the middle 95% of the differences. Both models score the same applications, so the comparison is paired.

In [9]:
def bootstrap_change(base_run, new_run, n=1000, seed=0):
    y, a, b = base_run["y_oot"], base_run["oot_scores"], new_run["oot_scores"]
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if y[i].min() != y[i].max():  # a draw needs both classes for an AUC
            diffs.append(roc_auc_score(y[i], b[i]) - roc_auc_score(y[i], a[i]))
    return np.percentile(diffs, [2.5, 97.5])

intervals = []
for d in CHECK_DATES:
    for m in MODELS:
        low, high = bootstrap_change(full[(d, m)], pruned[(d, m)])
        intervals.append({"train date": d, "model": m, "OOT applications": len(full[(d, m)]["y_oot"]),
                          "change": pruned[(d, m)]["oot"] - full[(d, m)]["oot"], "95% low": low, "95% high": high})
pd.DataFrame(intervals).round(4)

,train date,model,OOT applications,change,95% low,95% high
0,2024-01-01,logreg,1038,0.0084,0.0047,0.0124
1,2024-01-01,xgb,1038,0.0017,-0.0083,0.0124
2,2024-03-01,logreg,952,0.0008,-0.0011,0.0028
3,2024-03-01,xgb,952,-0.0005,-0.0070,0.0059
4,2024-05-01,logreg,941,0.0061,0.0016,0.0109
5,2024-05-01,xgb,941,-0.0053,-0.0147,0.0036


**Findings:**
- XGBoost's three intervals all include 0, so its changes can't be told apart from sampling noise.
- Logistic regression's gains at 2024-01-01 (+0.0084, interval +0.0047 to +0.0124) and 2024-05-01 (+0.0061, interval +0.0016 to +0.0109) exclude 0. The linear model is helped by losing columns that carry nothing. At 2024-03-01, its change (+0.0008) is within noise.

## 6. Decision

In [10]:
TOLERANCE = 0.005
within = bool((change["change"] >= -TOLERANCE).all())
if within:
    names = textwrap.fill(", ".join(f'"{c}"' for c in kept_cols["2024-03-01"]), 110, initial_indent="    ", subsequent_indent="    ", break_on_hyphens=False)
    decision = (f"ADOPT: the selected model loses at most {TOLERANCE} OOT AUC at every date (worst {change['change'].min():+.4f})\n"
                f"model_train.py uses the 2024-03-01 list, {len(kept_cols['2024-03-01'])} of {len(ALL_FEATURES)} columns:\n\n"
                f"FEATURE_COLUMNS = [\n{names},\n]")
else:
    decision = f"KEEP ALL 66: the selected model loses more than {TOLERANCE} OOT AUC at a train date (worst {change['change'].min():+.4f})"
print(decision)

ADOPT: the selected model loses at most 0.005 OOT AUC at every date (worst -0.0005)
model_train.py uses the 2024-03-01 list, 59 of 66 columns:

FEATURE_COLUMNS = [
    "Annual_Income", "Monthly_Inhand_Salary", "Num_Credit_Card", "Interest_Rate", "Num_of_Loan",
    "Delay_from_due_date", "Num_of_Delayed_Payment", "Changed_Credit_Limit", "Num_Credit_Inquiries",
    "Outstanding_Debt", "Credit_Utilization_Ratio", "Credit_History_Months", "Total_EMI_per_month",
    "Amount_invested_monthly", "Debt_to_Annual_Salary", "EMI_to_Monthly_Salary", "Occupation_Accountant",
    "Occupation_Architect", "Occupation_Developer", "Occupation_Doctor", "Occupation_Engineer",
    "Occupation_Entrepreneur", "Occupation_Journalist", "Occupation_Lawyer", "Occupation_Manager",
    "Occupation_Mechanic", "Occupation_Media_Manager", "Occupation_Musician", "Occupation_Scientist",
    "Occupation_Teacher", "Occupation_Writer", "Credit_Mix_Bad", "Credit_Mix_Standard", "Credit_Mix_Good",
    "Payment_of_Min_Amount_Y

**Findings:**
- The kept list meets the rule. `model_train.py` trains on the 59 columns kept at 2024-03-01, and the feature store keeps all 66.
- What it buys: 7 fewer features to monitor every month, with the deployed XGBoost unchanged within noise at 2024-03-01 (0.8563 → 0.8558). It doesn't buy a better deployed model at that date.
- For a refresh: the list was chosen on the 2024-03-01 training split, and other dates choose differently (section 3). A model trained at a later date should repeat this test on its own training split.
- For the deck: on the training split, neither candidate used 7 of the 66 features (permutation importance). Without them, the selected model stayed within 0.005 OOT AUC of the full model at three train dates (+0.0017, −0.0005, +0.0047). So the model uses 59 features, and monitoring tracks 7 fewer.